In [2]:
!pip install -q google-genai pandas numpy folium plotly

import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.express as px
import folium
from folium.plugins import HeatMap
from google import genai
from google.genai import types

# -------------------------------------------------------------------
# 1. INITIALISATION CLIENT & API (Compatible Local .env + Kaggle)
# -------------------------------------------------------------------
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Charge les variables du fichier .env (en local)
load_dotenv()

api_key = os.environ.get("GEMINI_API_KEY")

# 2. Si non trouvée dans .env/environnement, cherche dans Kaggle Secrets
if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        api_key = user_secrets.get_secret("GEMINI_API_KEY")
    except Exception:
        pass

# 3. Si toujours aucune clé trouvée, demande la saisie manuelle
if not api_key:
    api_key = input("🔑 Entrez votre clé API Google AI Studio : ")

client = genai.Client(api_key=api_key)
MODEL_ID = "gemma-2-9b-it"

print(f"✅ Client API Gemma initialisé avec le modèle : {MODEL_ID}")
# -------------------------------------------------------------------
# 2. GÉNÉRATION DU JEU DE DONNÉES MULTI-SOURCES (BUKAVU)
# -------------------------------------------------------------------
QUARTIERS = {
    "Kadutu": {"lat": -2.5050, "lon": 28.8450},
    "Ibanda": {"lat": -2.5180, "lon": 28.8600},
    "Bagira": {"lat": -2.4800, "lon": 28.8300},
    "Panzi":  {"lat": -2.5400, "lon": 28.8700}
}

MALADIES = ["Choléra", "Mpox", "Rougeole", "Paludisme grave"]
SOURCES = [
    ("Centre de Santé", 1.0),
    ("Pharmacie", 0.7),
    ("Relais Communautaire", 0.8),
    ("Signalement Citoyen", 0.4)
]

np.random.seed(42)
records = []
start_date = datetime.now() - timedelta(days=7)

for day in range(7):
    current_date = (start_date + timedelta(days=day)).strftime("%Y-%m-%d")
    for quartier, coords in QUARTIERS.items():
        # Cluster anormal de Choléra sur Kadutu
        is_outbreak = (quartier == "Kadutu" and day >= 4)
        num_signals = np.random.randint(12, 22) if is_outbreak else np.random.randint(2, 5)
        
        for _ in range(num_signals):
            idx = np.random.choice(len(SOURCES))
            source_name, weight = SOURCES[idx]
            maladie = "Choléra" if is_outbreak and np.random.rand() > 0.3 else np.random.choice(MALADIES)
            
            records.append({
                "date": current_date,
                "quartier": quartier,
                "latitude": coords["lat"] + np.random.normal(0, 0.0015),
                "longitude": coords["lon"] + np.random.normal(0, 0.0015),
                "maladie_suspectee": maladie,
                "source": source_name,
                "poids_confiance": weight,
                "nombre_cas": np.random.randint(1, 3)
            })

df_health = pd.DataFrame(records)
df_health['score_impact'] = df_health['nombre_cas'] * df_health['poids_confiance']
df_health.to_csv("donnees_sante_bukavu.csv", index=False)

print(f"✅ Jeu de données généré : {len(df_health)} signalements enregistrés.")

# -------------------------------------------------------------------
# 3. MOTEUR D'AGRÉGATION & CARTOGRAPHIE
# -------------------------------------------------------------------
synthesis = df_health.groupby(['quartier', 'maladie_suspectee']).agg(
    total_cas=('nombre_cas', 'sum'),
    score_confiance=('score_impact', 'sum'),
    sources_uniques=('source', 'nunique')
).reset_index().sort_values(by='score_confiance', ascending=False)

map_bukavu = folium.Map(location=[-2.5100, 28.8500], zoom_start=13, tiles="CartoDB positron")
heat_data = [[row['latitude'], row['longitude'], row['score_impact']] for _, row in df_health.iterrows()]
HeatMap(heat_data, radius=16, blur=11).add_to(map_bukavu)

for q, coords in QUARTIERS.items():
    cas_q = df_health[df_health['quartier'] == q]['nombre_cas'].sum()
    color = "red" if q == "Kadutu" else "blue"
    folium.CircleMarker(
        location=[coords['lat'], coords['lon']],
        radius=6 + (cas_q / 4),
        popup=f"Quartier: {q} | Cas: {cas_q}",
        color=color, fill=True, fill_color=color
    ).add_to(map_bukavu)

map_bukavu.save("carte_epidemie.html")
print("✅ Carte interactive enregistrée sous 'carte_epidemie.html'")

# -------------------------------------------------------------------
# 4. AGENT VEILLE SANITAIRE GEMMA API
# -------------------------------------------------------------------
def analyser_cluster_avec_gemma_api(quartier_cible):
    data_q = synthesis[synthesis['quartier'] == quartier_cible].to_json(orient="records")
    
    prompt = f"""
    Vous êtes EpiWatch AI, un assistant d'aide à la décision en veille épidémiologique pour la RDC.
    Examinez les données sanitaires récentes pour le quartier : {quartier_cible}.
    
    Données agrégées multi-sources (Centres de santé, Pharmacies, Relais communautaires, Citoyens) :
    {data_q}
    
    Veuillez fournir une analyse structurée en 3 points :
    1. **Évaluation du signal faible :** S'agit-il d'une anomalie/cluster ou d'un bruit statistique ? (Prenez en compte le poids de confiance des sources).
    2. **Hypothèses contextuelles :** Quelles raisons environnementales ou sanitaires locales pourraient expliquer cette hausse (ex. accès à l'eau, pluies) ? *Formulez des hypothèses, pas des certitudes absolues.*
    3. **Plan d'Action Réflexe :** Proposez 3 actions prioritaires et concrètes pour les relais communautaires et les autorités de santé.
    """
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2,
            max_output_tokens=600,
        )
    )
    
    return response.text

# Génération des rapports d'alerte pour Kadutu (zone rouge) et Ibanda (zone normale)
print("\n" + "="*50)
print("=== RAPPORT D'ANALYSE GEMMA API (KADUTU - ALERTE) ===")
print("="*50 + "\n")
print(analyser_cluster_avec_gemma_api("Kadutu"))

print("\n" + "="*50)
print("=== RAPPORT D'ANALYSE GEMMA API (IBANDA - CONTRÔLE) ===")
print("="*50 + "\n")
print(analyser_cluster_avec_gemma_api("Ibanda"))

ModuleNotFoundError: No module named 'kaggle_secrets'